# Automatic Differentiation with RustQuant

RustQuant provides a reverse-mode automatic differentiation (AAD) engine.
This is essential for computing sensitivities (Greeks) in quantitative finance,
as well as for optimization tasks like gradient descent.

## Key Concepts

1. **Graph**: A computation graph that records operations
2. **Variable**: A value tracked on the graph
3. **Accumulate**: Reverse-pass to compute all partial derivatives
4. **Gradient**: Access derivatives with `.wrt(&variable)`

## Setup

This notebook uses the [evcxr](https://github.com/evcxr/evcxr) Rust Jupyter kernel.

In [ ]:
:dep RustQuant = { path = "../crates/RustQuant" }

## 1. Simple Expressions

The basic workflow:
1. Create a `Graph`
2. Assign variables with `graph.var(value)`
3. Build expressions using standard math operators
4. Call `.accumulate()` to compute all gradients

In [ ]:
use RustQuant::autodiff::*;

let g = Graph::new();

let x = g.var(3.0);
let y = g.var(4.0);

// f(x, y) = x^2 + x*y + y^2
let f = x * x + x * y + y * y;

let gradient = f.accumulate();

println!("f(3, 4) = {}", f.value);
println!("df/dx   = {} (expected: 2*3 + 4 = 10)", gradient.wrt(&x));
println!("df/dy   = {} (expected: 3 + 2*4 = 11)", gradient.wrt(&y));

## 2. Transcendental Functions

The autodiff engine supports all standard math functions:
`sin`, `cos`, `tan`, `exp`, `ln`, `sqrt`, `powf`, `sinh`, `cosh`, `tanh`, etc.

In [ ]:
let g = Graph::new();

let x = g.var(1.0);
let y = g.var(2.0);

// f(x, y) = sin(x) * exp(y) + cos(x*y)
let f = x.sin() * y.exp() + (x * y).cos();

let grad = f.accumulate();

println!("f(1, 2)  = {:.6}", f.value);
println!("df/dx    = {:.6}", grad.wrt(&x));
println!("df/dy    = {:.6}", grad.wrt(&y));

## 3. Multivariate Functions

You can create many variables at once using `graph.vars()` and compute
gradients for all of them in a single reverse pass.

In [ ]:
// f(x, y, z) = x^(y + cos(1)) - atanh(z) / 2 + 1
fn my_function<'v>(vars: &[Variable<'v>], consts: &[f64]) -> Variable<'v> {
    vars[0].powf(vars[1] + consts[0].cos()) - 
    vars[2].atanh() / consts[1] +
    consts[0]
}

let graph = Graph::new();
let variables = graph.vars(&[3.0, 2.0, 0.5]);
let constants = [1.0, 2.0];

let result = my_function(&variables, &constants);
let gradient = result.accumulate();

println!("f(3, 2, 0.5) = {:.6}", result.value);
println!("Gradient: {:?}", gradient.wrt(&variables));
println!("Graph nodes: {}", graph.len());

## 4. Black-Scholes Greeks via Autodiff

One of the most powerful applications: compute all option Greeks simultaneously
with a single forward + reverse pass, instead of bumping each parameter individually.

In [ ]:
fn normcdf(x: Variable<'_>) -> Variable<'_> {
    0.5 * (-x / core::f64::consts::SQRT_2).erfc()
}

#[allow(non_snake_case)]
fn black_scholes_call<'v>(
    S: Variable<'v>,  // spot price
    K: Variable<'v>,  // strike
    T: Variable<'v>,  // time to maturity
    r: Variable<'v>,  // risk-free rate
    v: Variable<'v>,  // volatility
) -> Variable<'v> {
    let d1 = ((S / K).ln() + (r + v * v / 2.0) * T) / (v * T.sqrt());
    let d2 = d1 - v * T.sqrt();
    S * normcdf(d1) - K * (-r * T).exp() * normcdf(d2)
}

let graph = Graph::new();

let s = graph.var(100.0);  // Spot
let k = graph.var(100.0);  // Strike
let t = graph.var(1.0);    // 1 year
let r = graph.var(0.05);   // 5% rate
let v = graph.var(0.20);   // 20% vol

let call = black_scholes_call(s, k, t, r, v);
let greeks = call.accumulate();

println!("Call Price = {:.4}", call.value);
println!("Delta      = {:.4} (dC/dS)", greeks.wrt(&s));
println!("Rho        = {:.4} (dC/dr)", greeks.wrt(&r));
println!("Vega       = {:.4} (dC/dv)", greeks.wrt(&v));
println!("Theta      = {:.4} (dC/dT)", greeks.wrt(&t));

## 5. Visualizing the Computation Graph

RustQuant can export the computation graph in Graphviz DOT format,
which is useful for understanding and debugging complex expressions.

In [ ]:
let graph = Graph::new();
let vars = graph.vars(&[2.0, 3.0]);

let f = vars[0].sin() + vars[1].exp();
let _ = f.accumulate();

// Print the Graphviz DOT representation
println!("{}", graphviz(&graph, &vars));

## Summary

| Feature | Description |
|---------|-------------|
| Reverse-mode AD | Compute all partial derivatives in one pass |
| Operator overloading | Natural mathematical syntax |
| Full math support | Trig, exp, log, hyperbolic, power, etc. |
| Graphviz export | Visualize computation graphs |
| Finance applications | Greeks, sensitivities, calibration |